In [ ]:
import pandas as pd  
from scan_spectrolyser import scan_helpers
from scan_spectrolyser import no3_calibrations
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MultipleLocator
from hampel import hampel
import os



# Data Processing

## Correct for Turbidity

In [ ]:
dobson = scan_helpers.import_all_scan_fp('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/Dobson/Readings/', correct_dst=True, tz='US/Mountain')
dobson = dobson['2024-11-19':] # only use data after install date
dobson = dobson[(dobson.index <= '12/7/2024 10:30') | (dobson.index >= '2/19/25 00:00')] # drop data from period with bad scan
#dobson = scan_helpers.remove_invalid_abs(dobson, 45)
dobson_corrected = scan_helpers.correct_turbidity(dobson)
dobson

## Apply Calibrations

## Initial Calibrations

In [ ]:
dobson = scan_helpers.apply_calibrations(dobson, [no3_calibrations.one_wavelength, no3_calibrations.two_wavelength, no3_calibrations.second_derivative], {no3_calibrations.second_derivative: {'output_calibrated':False}, no3_calibrations.two_wavelength: {'output_calibrated':False}, no3_calibrations.one_wavelength: {'output_calibrated':False}})
dobson['two_wavelength_no3_mgl'] = (dobson['two_wavelength'] * 3.7649+.1904)* 14/1000
dobson['one_wavelength_no3_mgl'] = (dobson['one_wavelength']*3.6962-.6334) * 14/1000
dobson['second_derivative_no3_mgl'] = (dobson['second_derivative'] * 43.0111-.9440)* 14/1000

dobson_corrected = scan_helpers.apply_calibrations(dobson_corrected, [no3_calibrations.one_wavelength, no3_calibrations.two_wavelength, no3_calibrations.second_derivative], {no3_calibrations.second_derivative: {'output_calibrated':False}, no3_calibrations.two_wavelength: {'output_calibrated':False}, no3_calibrations.one_wavelength: {'output_calibrated':False}})
dobson_corrected['two_wavelength_no3_mgl_correct'] = (dobson_corrected['two_wavelength'] * 3.6742-.1730)* 14/1000
dobson_corrected['one_wavelength_no3_mgl_correct'] = (dobson_corrected['one_wavelength']*4.0634+.1136) * 14/1000
dobson_corrected['second_derivative_no3_mgl_correct'] = (dobson_corrected['second_derivative'] * 43.0231-.9425)* 14/1000
dobson_corrected

## After Insert Swap

In [ ]:
dobson.loc['2/6/2026':,'two_wavelength_no3_mgl'] = (dobson.loc['2/6/2026':,'two_wavelength'] * 4.0584-.0353)* 14/1000
dobson.loc['2/6/2026':,'one_wavelength_no3_mgl'] = (dobson.loc['2/6/2026':,'one_wavelength']*4.0385-.5123) * 14/1000
dobson.loc['2/6/2026':,'second_derivative_no3_mgl'] = (dobson.loc['2/6/2026':,'second_derivative'] * 50.0485+.0838)* 14/1000

dobson_corrected.loc['2/6/2026':,'two_wavelength_no3_mgl_correct'] = (dobson_corrected.loc['2/6/2026':,'two_wavelength'] * 3.9949-.1702)* 14/1000
dobson_corrected.loc['2/6/2026':,'one_wavelength_no3_mgl_correct'] = (dobson_corrected.loc['2/6/2026':,'one_wavelength']*4.4040+.3036) * 14/1000
dobson_corrected.loc['2/6/2026':,'second_derivative_no3_mgl_correct'] = (dobson_corrected.loc['2/6/2026':,'second_derivative'] * 50.0638+.0867)* 14/1000

dobson_corrected[['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl']] = dobson[['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl']]
dobson[['one_wavelength_no3_mgl_correct', 'two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct']] = dobson_corrected[['one_wavelength_no3_mgl_correct', 'two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct']]

dobson_corrected

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

dobson_corrected['11/1/2024':'12/10/24'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl','two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate Nov 2024 - July 2025', ax=ax, ylabel='Nitrate (mg/L)')
#rme.plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)
ax.yaxis.set_major_locator(MultipleLocator(.5))

## Cleanup Bad Values From Site Visits

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2024-12-04 09:30':'2024-12-04 12:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2024-12-04 09:30':'2024-12-04 12:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-02-19 00:00':'2025-02-20 00:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-02-19 00:00':'2025-02-20 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-02-23 00:00':'2025-02-23 03:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-02-23 00:00':'2025-02-23 03:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-02-25 12:00':'2025-02-26 03:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-02-25 12:00':'2025-02-26 03:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-03-11 00:00':'2025-03-12 00:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-03-11 00:00':'2025-03-12 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-03-25 00:00':'2025-03-26 00:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-03-25 00:00':'2025-03-26 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-04-01 00:00':'2025-04-03 00:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-04-01 00:00':'2025-04-03 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-04-17 00:00':'2025-04-18 00:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-04-17 00:00':'2025-04-18 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-05-01 00:00':'2025-05-03 00:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-05-01 00:00':'2025-05-03 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-05-12 00:00':'2025-05-13 00:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-05-12 00:00':'2025-05-13 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-05-22 00:00':'2025-05-23 00:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-05-22 00:00':'2025-05-23 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-05-30 00:00':'2025-05-31 00:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-05-30 00:00':'2025-05-31 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-06-10 09:40':'2025-06-10 10:20'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-06-10 09:40':'2025-06-10 10:20'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-06-26 00:00':'2025-06-27 10:20'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-06-26 00:00':'2025-06-27 10:20'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


- what is the jump on 6/26? hopefully just move to new location, not reblanking

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-07-09 00:00':'2025-07-9 13:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-07-09 00:00':'2025-07-9 13:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-07-23 00:00':'2025-07-23 13:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-07-23 00:00':'2025-07-23 13:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-08-05 00:00':'2025-08-05 14:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-08-05 00:00':'2025-08-05 14:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-08-19 00:00':'2025-08-19 13:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-08-19 00:00':'2025-08-19 13:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-08-29 09:00':'2025-08-29 15:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-08-29 09:00':'2025-08-29 15:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-09-09 12:00':'2025-09-09 15:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-09-09 12:00':'2025-09-09 15:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-09-18 06:00':'2025-09-18 16:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-09-18 06:00':'2025-09-18 16:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-09-30 00:30':'2025-09-30 18:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-09-30 00:30':'2025-09-30 18:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-10-16 12:30':'2025-10-16 14:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-10-16 12:30':'2025-10-16 14:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-10-31 11:00':'2025-10-31 14:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-10-31 11:00':'2025-10-31 14:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-11-20 09:00':'2025-11-20 11:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-11-20 09:00':'2025-11-20 11:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- a good illustration of why dobson needs drift correction.
- second derivative is less sensitive to drift than two wavelength, but it still shows up

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-12-04 12:15':'2025-12-04 13:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-12-04 12:15':'2025-12-04 13:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-12-30 10:00':'2025-12-30 13:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-12-30 10:00':'2025-12-30 13:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2026-01-13 10:00':'2026-01-13 13:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2026-01-13 10:00':'2026-01-13 13:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2026-01-27 14:00':'2026-01-27 18:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2026-01-27 14:00':'2026-01-27 18:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2026-02-06 12:00':'2026-02-06 18:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2026-02-06 12:00':'2026-02-06 18:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2026-02-18 12:00':'2026-02-18 18:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2026-02-18 12:00':'2026-02-18 18:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2026-02-25 12:00':'2026-02-25 18:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2026-02-25 12:00':'2026-02-25 18:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- dobson went crazy after Josh's 2-25 site visit until his next one.

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2026-03-11 09:00':'2026-03-11 18:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2026-03-11 09:00':'2026-03-11 18:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2026-03-23 08:00':'2026-03-23 18:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2026-03-23 08:00':'2026-03-23 18:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2026-04-06 08:00':'2026-04-06 18:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2026-04-06 08:00':'2026-04-06 18:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2026-04-21 08:00':'2026-04-21 18:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2026-04-21 08:00':'2026-04-21 18:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

### getting rid of dry period values (sensor out of water)

In [ ]:
fig, ax= plt.subplots()

dobson_corrected['2025-06-18 10:00':'2025-06-26 12:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
dobson_corrected['2025-06-18 10:00':'2025-06-26 12:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


In [ ]:
dobson_bad = [
    ('2024-12-04 09:30','2024-12-04 12:00'),
    ('2025-02-23 03:30', '2025-02-25 18:00'),
    ('2025-06-10 09:40','2025-06-10 10:20'),
    ('2025-06-18 10:00','2025-06-26 12:00'),
    ('2025-10-16 13:00','2025-10-16 14:00'),
    ('2025-12-04 12:15','2025-12-04 13:30'),
    
    
    
]
dobson_mask = np.array([
    (dobson.index >= start) & (dobson.index <= end)
    for start, end in dobson_bad
]).any(axis=0)

dobson_cleaned = dobson_corrected.loc[~dobson_mask]
dobson_cleaned_uncorrected = dobson.loc[~dobson_mask]
dobson_cleaned

### Replotting time series to see where absorbance threshold exceeded

In [ ]:
fig, ax = plt.subplots(figsize=(11, 11), nrows = 7, sharey=True)

#dobson_cleaned[dobson_cleaned_uncorrected[220.00] < 50].loc['2025-01-01':'2025-03-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[0], legend=False)
dobson_cleaned_uncorrected.loc['2025-01-01':'2025-03-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[0], legend=False)
ax[0].set_xlim(pd.to_datetime('2025-01-01'), pd.to_datetime('2025-03-01'))

#dobson_cleaned[dobson_cleaned_uncorrected[220.00] < 50].loc['2025-03-01':'2025-05-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[1], legend=False)
dobson_cleaned_uncorrected.loc['2025-03-01':'2025-05-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[1], legend=False)
ax[1].set_xlim(pd.to_datetime('2025-03-01'), pd.to_datetime('2025-05-01'))

#dobson_cleaned[dobson_cleaned_uncorrected[220.00] < 50].loc['2025-05-01':'2025-07-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[2], legend=False)
dobson_cleaned_uncorrected.loc['2025-05-01':'2025-07-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[2], legend=False)
ax[2].set_xlim(pd.to_datetime('2025-05-01'), pd.to_datetime('2025-07-01'))

#dobson_cleaned[dobson_cleaned_uncorrected[220.00] < 50].loc['2025-07-01':'2025-09-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[3], legend=False)
dobson_cleaned_uncorrected.loc['2025-07-01':'2025-09-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[3], legend=False)
ax[3].set_xlim(pd.to_datetime('2025-07-01'), pd.to_datetime('2025-09-01'))

#dobson_cleaned[dobson_cleaned_uncorrected[220.00] < 50].loc['2025-09-01':'2025-11-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[4], legend=False)
dobson_cleaned_uncorrected.loc['2025-09-01':'2025-11-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[4], legend=False)
ax[4].set_xlim(pd.to_datetime('2025-09-01'), pd.to_datetime('2025-11-01'))

#dobson_cleaned[dobson_cleaned_uncorrected[220.00] < 50].loc['2025-11-01':'2026-01-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[5], legend=False)
dobson_cleaned_uncorrected.loc['2025-11-01':'2026-01-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[5], legend=False)
ax[5].set_xlim(pd.to_datetime('2025-11-01'), pd.to_datetime('2026-01-01'))

#dobson_cleaned[dobson_cleaned_uncorrected[220.00] < 50].loc['2025-11-01':'2026-01-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[5], legend=False)
dobson_cleaned_uncorrected.loc['2026-01-01':'2026-03-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[6], legend=False)
ax[6].set_xlim(pd.to_datetime('2026-01-01'), pd.to_datetime('2026-03-01'))


for a in ax:
    a.set_xlabel(None)
    a.axhline(45, linestyle='--', color='r')
    a.tick_params(axis='x', rotation=0)
    a.yaxis.set_label_coords(-0.1, 0.5)
    for label in a.get_xticklabels():
        label.set_ha('center')

fig.suptitle('Dobson Absorbance at 220nm')

fig.tight_layout()


### Replotting the whole time series of cleaned values:

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

dobson_cleaned.plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')
#dobson.plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)
ax.yaxis.set_major_locator(MultipleLocator(.5))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

dobson_cleaned.plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')
#dobson.plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)
ax.yaxis.set_major_locator(MultipleLocator(.5))

- removing field visit data and absorbances above 45 is a pretty good start, but lowering absorbance threshold doesn't get rid of negative data in June

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

dobson_cleaned['2025-06-01':].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')
#dobson.plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)
ax.yaxis.set_major_locator(MultipleLocator(.5))

- data get bad around 6/17, but sensor was moved 6/26...
- ah this is probably the sensor not being submerged, take a look at the next set of data josh brings in and see it is fixed.

# Filter Outliers

In [ ]:
nitrate_cols = ['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl', 'one_wavelength_no3_mgl_correct', 'two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct']

filtered_nitrate = dobson_cleaned[nitrate_cols].apply(lambda x: hampel(x,window_size=10, n_sigma = 1.0).filtered_data, axis=0)
filtered_nitrate.index = dobson_cleaned.index
dobson_filtered = dobson_cleaned
dobson_filtered.loc[:, nitrate_cols] = filtered_nitrate


In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

dobson_filtered['02/23/25':].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate Feb-July 2025', ax=ax, ylabel='Nitrate (mg/L)')
#dobson_f['02/23/25':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)
#rme_cleaned['2/22 /2025':].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

dobson_filtered['06/01/25':].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Calculated Nitrate Feb-July 2025', ax=ax, ylabel='Nitrate (mg/L)')
#dobson_f['02/23/25':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)
#rme_cleaned['2/22 /2025':].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')


In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

dobson_filtered.loc['07/06/25'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'Dobson Thunderstorm 7/6/25', ax=ax, ylabel='Nitrate (mg/L)')
#dobson_f['02/23/25':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)
#rme_cleaned['2/22 /2025':].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')


# Plotting Feb 2026 Data To make sure insert swap good

In [ ]:
dobson_cleaned_uncorrected.loc[:,220.00:735.00].max(axis=1)

In [ ]:
fig, ax = plt.subplots(figsize=(11,8.5))

valid_abs = dobson_cleaned_uncorrected.loc[:,220.00:735.00].abs().max(axis=1) < 50
#dobson_cleaned_uncorrected['2026-01-08':][valid_abs].plot(y=[220.00, 265.00], ylabel='Absorbance', ax=ax, label = ['Corrected 220nm','Corrected 265nm'], secondary_y=True)
#rme_cleaned_uncorrected['2026-01-01':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Uncorrected 220nm','Uncorrected 265nm'])
dobson_cleaned['2026-04-01':][valid_abs].plot(y=['second_derivative_no3_mgl_correct', 'two_wavelength_no3_mgl_correct'], ylabel='Absorbance', ax=ax, label = ['Corrected Second Derivative','Corrected Two Wavelength'])


- step change in second derivative after insert swap, but continues data from two wavelength
- things go bananas after 2/22, I think that must be air bubbles from aeration and big storm.. 
- did josh move the s::can? looks like maybe aroudn 3/8
- despite the blanking issues, data from dobson is way better than RME. 

In [ ]:
fig, ax = plt.subplots(figsize=(11,8.5))

valid_abs = dobson_cleaned_uncorrected.loc[:,220.00:735.00].abs().max(axis=1) < 50.0
#dobson_cleaned['2026-01-08':][valid_abs].plot(y=[220.00, 265.00], ylabel='Absorbance', ax=ax, label = ['Corrected 220nm','Corrected 265nm'], secondary_y=True)
#rme_cleaned_uncorrected['2026-01-01':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Uncorrected 220nm','Uncorrected 265nm'])
dobson_cleaned['2026-02-01':][valid_abs].plot(y=['second_derivative_no3_mgl_correct', 'two_wavelength_no3_mgl_correct'], ylabel='Absorbance', ax=ax, label = ['Corrected Second Derivative','Corrected Two Wavelength'])


In [ ]:
fig, ax = plt.subplots(figsize=(11,8.5))

dobson_cleaned['2026-01-08':'2026-02-22'].plot(y=[220.00, 265.00], ylabel='Absorbance', ax=ax, label = ['Corrected 220nm','Corrected 265nm'], secondary_y=True)
#rme_cleaned_uncorrected['2026-01-01':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Uncorrected 220nm','Uncorrected 265nm'])
dobson_cleaned['2026-01-08':'2026-02-22'].plot(y=['second_derivative_no3_mgl_correct', 'two_wavelength_no3_mgl_correct'], ylabel='Absorbance', ax=ax, label = ['Corrected Second Derivative','Corrected Two Wavelength'])


In [ ]:
fig, ax = plt.subplots(figsize=(11,8.5))

dobson_cleaned['2026-03-01':].plot(y=[220.00, 265.00], ylabel='Absorbance', ax=ax, label = ['Corrected 220nm','Corrected 265nm'], secondary_y=True)
#rme_cleaned_uncorrected['2026-01-01':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Uncorrected 220nm','Uncorrected 265nm'])
dobson_cleaned['2026-03-01':].plot(y=['second_derivative_no3_mgl_correct', 'two_wavelength_no3_mgl_correct'], ylabel='Absorbance', ax=ax, label = ['Corrected Second Derivative','Corrected Two Wavelength'])


In [ ]:
fig, ax = plt.subplots(figsize=(11,8.5))

dobson_cleaned['2026-01-08':'2026-02-23'].plot(y=[220.00, 265.00], ylabel='Absorbance', ax=ax, label = ['Corrected 220nm','Corrected 265nm'], secondary_y=True)
#rme_cleaned_uncorrected['2026-01-01':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Uncorrected 220nm','Uncorrected 265nm'])
dobson_cleaned['2026-01-08':'2026-02-23'].plot(y=['second_derivative_no3_mgl_correct', 'two_wavelength_no3_mgl_correct'], ylabel='Absorbance', ax=ax, label = ['Corrected Second Derivative','Corrected Two Wavelength'])


- there is a pretty large step change in absorbance and second derivative no3 after insert swap. this could just be that the lens was fouled to begin with
- need to assess how new calibratipn matches field samples and also possibly adjust bias term
- new data look a little less noisy though which is good

In [ ]:
fig, ax = plt.subplots(figsize=(11,8.5))

#dobson_cleaned['2026-03-10':].plot(y=[220.00, 265.00], ylabel='Absorbance', ax=ax, label = ['Corrected 220nm','Corrected 265nm'], secondary_y=True)
dobson_cleaned_uncorrected['2026-03-10':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Uncorrected 220nm','Uncorrected 265nm'])
dobson_cleaned['2026-03-10':].plot(y=['second_derivative_no3_mgl_correct', 'two_wavelength_no3_mgl_correct'], ylabel='Absorbance', ax=ax, label = ['Corrected Second Derivative','Corrected Two Wavelength'])


- data after the 3/10 look pretty good - good agreement b/w second derivative and two wavelength, and fairly large diurnal variation. Too bad we missed the big peak, hopefully caught that on the autosampler.

In [ ]:
fig, ax = plt.subplots(figsize=(11,8.5), nrows=2)
dobson_cleaned['2026-02-23':].plot(y=['second_derivative_no3_mgl_correct', 'two_wavelength_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[0], label = ['Corrected Second Derivative','Corrected Two Wavelength'], title= 'Dobson ROS Event Feb 2026')

dobson_cleaned_uncorrected['2026-02-23':].plot(y=[220.00, 265.00], ylabel='Absorbance', ax=ax[1], label = ['Corrected 220nm','Corrected 265nm'], secondary_y=True)
#dobson_cleaned_uncorrected['2026-01-01':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Uncorrected 220nm','Uncorrected 265nm'])


# Export

In [ ]:
dobson_filtered.to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/Dobson/Processed Data/dobson_cleaned.csv')